In [ ]:
%pip install pytest==8.4.2

In [ ]:
%restart_python

## ⚠️ Important Notice: Use a Personal Cluster for Structured Streaming

**Attention students:**  

This notebook uses **Structured Streaming** features that are **not fully supported in serverless clusters**.  

To ensure full functionality, including:   
- Continuous streaming processing  
- Checkpointing and state management  

You **must use a personal (interactive) cluster** or a standard Databricks cluster with the correct runtime.  

> ❌ Serverless clusters may limit streaming capabilities, prevent proper checkpointing, and can cause runtime errors.  

**Recommended actions:**  
1. Start a **personal cluster** in your workspace.  
2. Make sure the cluster uses a **Databricks Runtime version that supports Structured Streaming** (preferably latest LTS).  
3. Attach this notebook to the personal cluster before running or creating your DLT pipeline.  

Following this ensures your streaming pipelines work correctly and all transformations execute as expected.


> **Key Note:**
> During the notebook you will find a series of function definitions. They're only a guide, so feel free to use or remove them, and create your own pieces of code.

## 🎯 What you will learn in this lab

By the end of this notebook you should be able to explain, in your own words:

1. **Auto Loader basics** — how `cloudFiles` discovers new files and turns a directory into a stream.
2. **Schema inference & the schema location** — where Auto Loader remembers the schema, and why it versions it.
3. **Schema evolution** — what `addNewColumns` actually does when an unexpected column shows up (spoiler: the stream *fails on purpose*).
4. **The `_rescued_data` column** — how Auto Loader avoids silently dropping data it did not expect.
5. **Checkpoints & exactly-once** — why re-running a failed stream does *not* duplicate rows.
6. **Triggers** — the difference between `availableNow` (batch-style) and `processingTime` (continuous micro-batches).
7. **Monitoring** — how to read `StreamingQuery` progress metrics.

> 💡 Work through the cells in order. Several sections depend on state (checkpoints, schema files) created by earlier ones.

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.streaming import StreamingQuery
from typing import Optional
import json
from helpers import utils, generate_streaming_events as gen_stream

## 🗂️ Defining Paths and Environment Variables

In this step, we will **set up the environment and directory structure** that will be used during the streaming process.

- `capstone_env`: defines the environment (e.g., `dev`, `prod`), making the code reusable across different stages.  
- `base_schema`: retrieves the base schema name using the helper function `utils.get_base_user_schema()`.  
- `volume_base_path`: base directory path where all data and metadata related to the stream will be stored.  
- `raw_path`: location where the raw event files will be written.  
- `checkpoint_path`: location where Spark will save checkpoints to keep track of stream progress.  
- `schema_location_path`: directory where the evolving schema will be stored.  
- `output_table`: name of the Delta table where the processed events will be written.

We also print the paths to verify that they were set correctly.


In [ ]:
catalog = utils.get_param("catalog", "capstone_dev")
env = utils.get_param("env", "dev")
bronze_schema = f"{utils.get_base_user_schema()}_bronze"

volume_base_path = f"/Volumes/{catalog}/{bronze_schema}"
raw_path = f"{volume_base_path}/raw_files/web_site_events"
checkpoint_path = f"{volume_base_path}/checkpoint_files/web_site_events"
schema_location_path = f"{volume_base_path}/schema_files/web_site_events"
output_table = f"{catalog}.{bronze_schema}.web_site_events"

# Paths used only by the optional / advanced sections at the end of the notebook.
# They get their own checkpoint + schema location so that the experiments there
# cannot corrupt the state of the main exercise above.
sandbox_checkpoint_path = f"{volume_base_path}/checkpoint_files/sandbox_web_site_events"
sandbox_schema_location_path = f"{volume_base_path}/schema_files/sandbox_web_site_events"
sandbox_table = f"{catalog}.{bronze_schema}.web_site_events_sandbox"

print(f"RAW PATH: {raw_path}")
print(f"CHECKPOINT: {checkpoint_path}")
print(f"SCHEMA LOCATION: {schema_location_path}")
print(f"TABLE: {output_table}")

## ⚠️ Two very different things are called "schema evolution" in this lab

Read this before you run anything — it is the single most common point of confusion.

| | `enable_schema_evolution` (generator) | `cloudFiles.schemaEvolutionMode` (Auto Loader) |
|---|---|---|
| **Where it lives** | `gen_stream.stream_events(...)` argument | `.option(...)` on the read stream |
| **What it controls** | Whether the **source JSON files** contain extra fields (e.g. `referrer_url`) | How **Auto Loader reacts** when it meets a field it has never seen |
| **Changes in this lab?** | Yes — `False` first, then `True` | No — stays `addNewColumns` for the main exercise |

In short: the generator flag changes **the data**. The Auto Loader option changes **the reaction to the data**.
You are not toggling the same switch twice.

## 📥 Defining a Function to Read Streaming Data (JSON)

In this step, we define a helper function called `read_stream_json` to **read JSON files in streaming mode** using **Auto Loader**.

### Function details:
- `source_path`: the path to the directory containing the raw JSON files.
- `schema_path`: the path where Auto Loader stores the inferred schema so it can track changes over time.
- `evolution_mode`: how Auto Loader reacts to columns it has never seen before (default `addNewColumns`).
- `max_files_per_trigger`: caps how many files are pulled into a single micro-batch.

### Key options:
- `cloudFiles.format = "json"` → defines the file format.
- `cloudFiles.inferColumnTypes = true` → automatically detects column types instead of reading everything as string.
- `cloudFiles.schemaEvolutionMode = addNewColumns` → new columns are added to the schema, **after the stream fails once on purpose** (more on this later).
- `cloudFiles.schemaLocation` → directory where Auto Loader stores versioned schema information.
- `cloudFiles.rescuedDataColumn = "_rescued_data"` → any field that could not fit the current schema is captured here as JSON instead of being dropped.
- `cloudFiles.maxFilesPerTrigger` → back-pressure control: limits micro-batch size so one batch cannot swallow the whole backlog.
- `multiLine = true` → supports JSON files spanning multiple lines.

> 🔎 **Why `_rescued_data` matters:** without it, a value that does not match the inferred type (a string where an int was expected, a nested field that changed shape) is simply lost. With it, the raw JSON is preserved in that column so you can detect and repair the problem downstream. In production this is your early-warning system for upstream schema drift.

In [ ]:
def read_stream_json(
    source_path: str,
    schema_path: str,
    evolution_mode: str = "addNewColumns",
    max_files_per_trigger: int = 50,
) -> DataFrame:
    """
    This function returns a streaming DataFrame that can be used for further transformations or writes.
    Args:
        source_path (str): Path to the directory containing the source JSON files.
        schema_path (str): Path to the schema location used by Auto Loader to track and evolve the schema.
        evolution_mode (str): One of "addNewColumns", "rescue", "failOnNewColumns", "none".
        max_files_per_trigger (int): Maximum number of files processed in a single micro-batch.
    Returns:
        DataFrame: A streaming DataFrame representing the ingested JSON data.
    """
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", evolution_mode)
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.maxFilesPerTrigger", str(max_files_per_trigger))
        .option("multiLine", "true")
        .load(source_path)
    )

## 💾 Defining a Function to Write Streaming Data to Delta Lake

In this step, we define the helper function `write_stream_to_delta` to **write streaming data to a Delta Lake table** with **schema evolution enabled** through the `mergeSchema` option.

### Function details:
- `df`: the streaming DataFrame that will be written to Delta Lake.
- `checkpoint_path`: the directory where Spark stores checkpoints to maintain stream state.
- `table_name`: the name of the target Delta table.
- `processing_time`: if provided (e.g. `"10 seconds"`), the query runs **continuously**; if `None`, it uses `availableNow`.

### Key options:
- `format("delta")` → writes the stream in Delta Lake format.
- `checkpointLocation` → stores metadata and progress information for fault tolerance.
- `mergeSchema = true` → allows Delta Lake to automatically update the table schema as new columns appear.
- `outputMode = append` → continuously appends new records to the table.

### Triggers
| Trigger | Behaviour | Typical use |
|---|---|---|
| `availableNow=True` | Processes everything currently available in as many micro-batches as needed, then **stops** | Development, scheduled/incremental jobs |
| `processingTime="X seconds"` | Fires a micro-batch every X seconds and **keeps running** until stopped | Always-on production pipelines |
| *(no trigger)* | Micro-batches fire back-to-back as fast as possible | Lowest latency, hardest to reason about |

> 🔐 **Checkpoint = exactly-once.** The checkpoint directory records exactly which source files have been committed to the table. Delta commits the data and the offset atomically, so if the job crashes halfway you can simply re-run it: already-committed files are skipped and rows are **not** duplicated. You will verify this yourself in a moment. This is also why *deleting the checkpoint means reprocessing everything from scratch*.

In [ ]:
def write_stream_to_delta(
    df: DataFrame,
    checkpoint_path: str,
    table_name: str,
    processing_time: Optional[str] = None,
) -> StreamingQuery:
    """
    This function returns a `StreamingQuery` object that represents the active streaming job.
    Args:
        df (DataFrame): The streaming DataFrame to be written to Delta Lake.
        checkpoint_path (str): Path to the directory for storing checkpoint data.
        table_name (str): Name of the target Delta Lake table.
        processing_time (Optional[str]): If set, run continuously with this trigger interval
            (e.g. "10 seconds"). If None, use trigger(availableNow=True).
    Returns:
        StreamingQuery: The active streaming query writing data to Delta Lake.
    """
    writer = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
    )

    writer = (
        writer.trigger(processingTime=processing_time)
        if processing_time
        else writer.trigger(availableNow=True)
    )

    return writer.toTable(table_name)

## 📊 A small helper to inspect what the stream actually did

A streaming query is not a black box. Every micro-batch emits a progress report you can read
programmatically — the same numbers you see in the **Structured Streaming** tab of the Spark UI
(open the cell's *View → Spark UI* link while a query is running).

The metrics worth knowing:
- `numInputRows` → rows read in that micro-batch.
- `batchId` → monotonically increasing micro-batch counter.
- `durationMs` → time spent in each phase (planning, execution, commit).
- `sources[0].description` → what was read (for Auto Loader, the file paths and offsets).

In [ ]:
def print_stream_progress(query: StreamingQuery, show_last_batch: bool = True) -> None:
    """Summarise what a (finished or running) streaming query has processed so far."""
    progress = query.recentProgress

    if not progress:
        print("No progress reported yet — the query may not have run a micro-batch.")
        return

    total_rows = sum(p.get("numInputRows", 0) for p in progress)
    print(f"Query name/id : {query.name} / {query.id}")
    print(f"Micro-batches : {len(progress)}")
    print(f"Rows ingested : {total_rows:,}")
    print(f"Still active  : {query.isActive}")

    print("\nPer-batch summary:")
    for p in progress:
        print(
            f"  batch {p['batchId']:>3} | "
            f"rows={p.get('numInputRows', 0):>8,} | "
            f"duration={p.get('batchDuration', 0):>6} ms | "
            f"rows/s={p.get('processedRowsPerSecond', 0):,.0f}"
        )

    if show_last_batch and query.lastProgress:
        print("\nFull last-batch progress report:")
        print(json.dumps(query.lastProgress, indent=2)[:2000])

## 🚀 Running the Streaming Job with the Initial Schema

In this step, we **generate source files that only contain the original fields**, and then ingest them.

### Instructions:
1. **Open the simulation script** at `helpers/generate_streaming_events.py` and skim it, so you know what the data looks like.

2. Run the generator below with `enable_schema_evolution=False`.
   This means the **generated JSON files contain no extra columns** — it does *not* change any Auto Loader setting.

*(Optional)* Adjust the following parameters to test different scenarios:

- `batch_size`: number of events per batch
- `delay_seconds`: delay between batches
- `total_batches`: total number of batches
- `null_frequency`: frequency of null values in the data

### What the ingestion cell does:
- Reads streaming JSON data from `raw_path` using `read_stream_json`.
- Prints the schema Auto Loader inferred — **take a screenshot or note the columns, you will compare later.**
- Writes to a Delta table with checkpointing.
- Waits for the query to finish, then prints its progress metrics.

In [ ]:
gen_stream.stream_events(
    batch_size=10000,
    delay_seconds=1,
    total_batches=10,
    null_frequency=500,
    env=env,
    enable_schema_evolution=False
)

In [ ]:
df_stream = read_stream_json(raw_path, schema_location_path)

print("=== Schema inferred by Auto Loader (run 1) ===")
df_stream.printSchema()

query = write_stream_to_delta(df_stream, checkpoint_path, output_table)
query.awaitTermination()

print("\n=== Streaming progress ===")
print_stream_progress(query, show_last_batch=False)

## ✅ Validating the Delta Table

After running the streaming job, it's important to **verify that the data was correctly written** to the Delta table.

The following command:

- Executes a simple SQL query on the Delta table defined by `output_table`.  
- Retrieves the first 100 records to give a quick preview of the data.  
- Displays the results in a tabular format for easy inspection.


In [ ]:
spark.sql(f"SELECT * FROM {output_table} LIMIT 100").display()
spark.sql(f"SELECT COUNT(*) AS row_count FROM {output_table}").display()

## 🔐 Experiment: checkpoints give you exactly-once, so re-running is safe

Before moving on, prove the guarantee to yourself instead of taking it on faith.

The cell below re-runs **the exact same stream, against the exact same files, with the exact same checkpoint**.
Predict the result before you run it:

- ❓ Will the row count double?
- ❓ How many rows will the micro-batches report?

Then run it and check your prediction.

> 💡 The checkpoint stores which files have already been committed. A re-run finds no *new* files, so it
> commits zero rows. This is precisely why a failed streaming job can just be restarted.

In [ ]:
count_before = spark.sql(f"SELECT COUNT(*) AS c FROM {output_table}").collect()[0]["c"]

df_stream = read_stream_json(raw_path, schema_location_path)
query = write_stream_to_delta(df_stream, checkpoint_path, output_table)
query.awaitTermination()

count_after = spark.sql(f"SELECT COUNT(*) AS c FROM {output_table}").collect()[0]["c"]

print(f"Rows before re-run : {count_before:,}")
print(f"Rows after re-run  : {count_after:,}")
print(f"Duplicated rows    : {count_after - count_before:,}  <-- expected 0")
print()
print_stream_progress(query, show_last_batch=False)

## 🚀 Running the Streaming Job with New Columns in the Source

Now we make the **source data change**: the generator starts emitting an extra field (`referrer_url`).
Auto Loader is still configured exactly as before (`schemaEvolutionMode = addNewColumns`).

### Instructions:
1. Run the generator below with `enable_schema_evolution=True`.
2. Then run the ingestion cell **and read the output carefully** — it is *supposed* to fail the first time.

### ✋ Predict first
Before running the ingestion cell, write down your answer:

- Will the stream succeed, fail, or silently drop the new column?
- If it fails, what state will the table and the checkpoint be left in?
- How many versions do you expect to find under the schema location?

*(Optional)* Adjust the same simulation parameters as before.

In [ ]:
gen_stream.stream_events(
    batch_size=10000,
    delay_seconds=1,
    total_batches=10,
    null_frequency=500,
    env=env,
    enable_schema_evolution=True
)

In [ ]:
# This run is EXPECTED to fail the first time. We catch the exception so you can read it,
# instead of just seeing a red stack trace.

df_stream = read_stream_json(raw_path, schema_location_path)

print("=== Schema Auto Loader is starting from ===")
df_stream.printSchema()

try:
    query = write_stream_to_delta(df_stream, checkpoint_path, output_table)
    query.awaitTermination()
    print("\n✅ Stream completed without errors.")
    print_stream_progress(query, show_last_batch=False)
except Exception as e:
    print("\n❌ The stream stopped with an exception — this is the expected behaviour:")
    print(f"\nException type: {type(e).__name__}")
    print(f"\nMessage:\n{e}")
    print(
        "\n👉 Read the message above. Auto Loader detected a column it had never seen, "
        "recorded the NEW schema in the schema location, and then failed the query on purpose "
        "so that a restart picks up the updated schema."
    )

### Did the read/write fail? Good — that is the lesson.

> **Note:**
> The default behaviour of Auto Loader in `addNewColumns` mode is:
>
> - The new column **is written to the schema location** as a new schema version.
> - The stream then **fails deliberately** so it can restart with that new schema.
> - Existing columns **do not** change data type (a type change is rescued, not evolved).
>
> So you must **re-run** the read/write to see the new column reflected in the destination table.
>
> Inspect the schema location — you should now see **two schema versions**. The cell below lists them
> and diffs them so you can see exactly what changed.
>
> On a scheduled Databricks job this restart is automatic (configure the task to retry), which is why
> the behaviour is considered "self-healing" rather than an outage.
>
> See more: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#how-does-auto-loader-schema-evolution-work

In [ ]:
# Inspect the versioned schemas Auto Loader keeps in the schema location.
schemas_dir = f"{schema_location_path}/_schemas"

def _version_key(f):
    try:
        return int(f.name.rstrip("/"))
    except ValueError:
        return -1

versions = sorted(dbutils.fs.ls(schemas_dir), key=_version_key)
print(f"Schema versions in {schemas_dir}: {[v.name for v in versions]}")
print("More than one version means Auto Loader evolved the schema.\n")

for v in versions:
    print(f"--- version {v.name} ---")
    print(dbutils.fs.head(v.path, 4096))
    print()

# TODO: compare the first and last version. Which column appears only in the last one?

### 🔁 Now re-run the stream to pick up the evolved schema

In [ ]:
df_stream = read_stream_json(raw_path, schema_location_path)

print("=== Schema after evolution ===")
df_stream.printSchema()

query = write_stream_to_delta(df_stream, checkpoint_path, output_table)
query.awaitTermination()

print("\n=== Streaming progress ===")
print_stream_progress(query, show_last_batch=False)

## ✅ Validating the Delta Table After Schema Evolution

After running the streaming job with **schema evolution enabled**, it's important to **verify that new columns were correctly added** to the Delta table.

The following command:

- Executes a SQL query on the Delta table defined by `output_table`.  
- Filters for records where the newly added column `referrer_url` is not null, helping to confirm that schema evolution worked.  
- Retrieves the first 20 records to quickly inspect the updated schema and data.  
- Displays the results in a tabular format for easy observation of the new column.


In [ ]:
spark.sql(f"SELECT * FROM {output_table} WHERE referrer_url IS NOT NULL LIMIT 20").display()
spark.sql(f"SELECT COUNT(*) AS row_count FROM {output_table}").display()

# Rows ingested BEFORE the evolution still have NULL in the new column - Delta backfills nothing.
spark.sql(f"""
    SELECT
        referrer_url IS NULL AS ingested_before_evolution,
        COUNT(*) AS rows
    FROM {output_table}
    GROUP BY 1
""").display()

## 🛟 Inspecting `_rescued_data`

`_rescued_data` is Auto Loader's safety net. A record lands there when a field:

- does **not** match the type in the current schema (e.g. a string arriving where an integer was inferred), or
- appears in the file but is **not** in the schema Auto Loader is currently using.

The value is stored as raw JSON, together with the source file path, so **nothing is silently lost**.

### 🔎 What to look for
- Some rows should have a non-null `_rescued_data` containing `referrer_url` — those are the rows read
  in the micro-batch *before* the schema evolved.
- In production, a query like the one below is exactly what you would alert on: a sudden rise in rescued
  rows means the upstream producer changed something.

In [ ]:
rescued = spark.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_rows
    FROM {output_table}
""")
rescued.display()

spark.sql(f"""
    SELECT _rescued_data, *
    FROM {output_table}
    WHERE _rescued_data IS NOT NULL
    LIMIT 20
""").display()

## 🧪 Tests

The suite below validates the table you just built. Broadly it checks that:

- the table exists and is non-empty,
- the expected bronze columns are present, including the column added by schema evolution,
- data types match what Auto Loader should have inferred,
- null handling matches what the generator produced.

**TO DO:**
- Run the tests, check whether any of them fail, and investigate the **root cause** of each failure.
- A failure is usually a symptom of a step above being skipped or run out of order (for example, not
  re-running the stream after the deliberate schema-evolution failure). Read the assertion message,
  then go back and confirm the state of the table, the checkpoint and the schema location.

In [ ]:
from helpers import test_runner
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
os.environ["NOTEBOOK_NAME"] = notebook_path.split("/")[-1]

test_runner.run(
    df=spark.sql(f"SELECT * FROM {output_table}"),
    schema_evoluted=True
)

---
# 🧭 Optional / advanced exercises

Everything below is **sandboxed**: it writes to `sandbox_table` with its own checkpoint and schema
location, so it cannot break the table your tests just validated.

⚠️ These streams start from an empty checkpoint, so they **reprocess every file** in `raw_path`.
Keep the generator volumes small if you are on a modest cluster.

## 🧪 Exercise A — compare the other `schemaEvolutionMode` values

You have only seen `addNewColumns`. Auto Loader supports four modes:

| Mode | Behaviour when an unknown column appears |
|---|---|
| `addNewColumns` (default) | Adds the column to the schema and **fails the stream** so it restarts with the new schema |
| `rescue` | Schema never changes; unknown columns go to `_rescued_data`. The stream **never fails** |
| `failOnNewColumns` | Stream fails and **does not** update the schema — you must fix it manually |
| `none` | Unknown columns are ignored entirely and **not** rescued |

Run the cell below with `evolution_mode="rescue"` against the *same* evolved files and compare:

- ❓ Does the stream fail?
- ❓ Does `referrer_url` appear as its own column, or inside `_rescued_data`?
- ❓ Which mode would you choose for a pipeline that must never stop, and what do you give up?

Then change the mode to `failOnNewColumns` (delete the sandbox checkpoint first, see the cleanup cell)
and observe the difference.

In [ ]:
MODE = "rescue"  # try also: "addNewColumns", "failOnNewColumns", "none"

df_sandbox = read_stream_json(
    raw_path,
    sandbox_schema_location_path,
    evolution_mode=MODE,
)

print(f"=== Schema with evolution_mode='{MODE}' ===")
df_sandbox.printSchema()

try:
    q = write_stream_to_delta(df_sandbox, sandbox_checkpoint_path, sandbox_table)
    q.awaitTermination()
    print(f"\n✅ Stream finished without failing in mode '{MODE}'.")
    print_stream_progress(q, show_last_batch=False)
except Exception as e:
    print(f"\n❌ Stream failed in mode '{MODE}': {type(e).__name__}")
    print(e)

In [ ]:
# Where did referrer_url end up in the sandbox table?
spark.sql(f"DESCRIBE TABLE {sandbox_table}").display()

spark.sql(f"""
    SELECT _rescued_data
    FROM {sandbox_table}
    WHERE _rescued_data IS NOT NULL
    LIMIT 10
""").display()

## ⏱️ Exercise B — a continuously running stream (`processingTime`)

Everything so far used `trigger(availableNow=True)`: process the backlog, then stop. Production
pipelines usually run **continuously**. The difference is easiest to feel by doing it.

Run the next three cells **in order, without waiting for the first one to "finish"** — it will not finish,
that is the point:

1. **Start** a stream with `processingTime="10 seconds"`. The cell returns immediately; the query keeps
   running in the background.
2. **Generate new files** while it is running. Watch the row count grow between micro-batches.
3. **Stop** the query explicitly with `query.stop()`.

### ❓ Questions to answer
- How does `query.status` change between micro-batches?
- What does a micro-batch report when *no* new files arrived?
- What happens to the checkpoint if you stop the query mid-batch and restart it?

In [ ]:
# 1) Start the continuous query (returns immediately - it runs in the background)
continuous_query = write_stream_to_delta(
    read_stream_json(raw_path, sandbox_schema_location_path, evolution_mode="rescue"),
    f"{sandbox_checkpoint_path}_continuous",
    f"{sandbox_table}_continuous",
    processing_time="10 seconds",
)

print(f"Query started. isActive = {continuous_query.isActive}")
print(f"Status: {continuous_query.status}")

In [ ]:
# 2) Drop new files into the source WHILE the query above is still running.
gen_stream.stream_events(
    batch_size=1000,
    delay_seconds=2,
    total_batches=5,
    null_frequency=500,
    env=env,
    enable_schema_evolution=True,
)

In [ ]:
# 3) Watch the micro-batches, then stop the query.
import time

for _ in range(3):
    print(f"status={continuous_query.status['message']!r}")
    if continuous_query.lastProgress:
        lp = continuous_query.lastProgress
        print(f"  batchId={lp['batchId']} rows={lp.get('numInputRows', 0):,}")
    time.sleep(10)

print_stream_progress(continuous_query, show_last_batch=False)

continuous_query.stop()
print(f"\nStopped. isActive = {continuous_query.isActive}")

In [ ]:
# Safety net: stop every streaming query still running in this session.
for q in spark.streams.active:
    print(f"Stopping active query: {q.name} ({q.id})")
    q.stop()

print(f"Active queries remaining: {len(spark.streams.active)}")

---
## 🧹 Reset / cleanup

Run this **only** when you want to start the lab over from a clean slate, or to free up storage
when you are done.

Deleting the checkpoint and the schema location makes Auto Loader forget everything: the next run
reprocesses every file in `raw_path` and re-infers the schema from scratch. That is the correct way
to "rewind" this lab — editing the Delta table alone is not enough, because the checkpoint would still
consider the old files as already processed.

> ⚠️ Set `CONFIRM_RESET = True` to actually run it. It is disabled by default so you cannot wipe your
> work with a stray *Run All*.

In [ ]:
CONFIRM_RESET = False  # set to True to actually delete

def _safe_rm(path: str) -> None:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"removed: {path}")
    except Exception as e:
        print(f"skipped: {path} ({type(e).__name__})")

if CONFIRM_RESET:
    for q in spark.streams.active:
        q.stop()

    for tbl in (
        output_table,
        sandbox_table,
        f"{sandbox_table}_continuous",
    ):
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")
        print(f"dropped table: {tbl}")

    for p in (
        checkpoint_path,
        schema_location_path,
        sandbox_checkpoint_path,
        f"{sandbox_checkpoint_path}_continuous",
        sandbox_schema_location_path,
    ):
        _safe_rm(p)

    # Uncomment to also delete the generated source files:
    # _safe_rm(raw_path)

    print("\nReset complete. Re-run the notebook from the top.")
else:
    print("CONFIRM_RESET is False - nothing was deleted.")

## 📝 Wrap-up — check your understanding

Answer these without scrolling back up. If any of them is fuzzy, revisit the matching section.

1. You delete the checkpoint directory but keep the Delta table. What happens on the next run, and why is that a problem?
2. `addNewColumns` "fails the stream". Why is that a *feature* rather than a bug, and what makes it safe?
3. A field arrives as `"42"` (string) where the schema says `bigint`. Which mode puts it in `_rescued_data`, and which one drops it?
4. Your table has 1,000,000 rows and `maxFilesPerTrigger = 50`. What does that setting actually control, and when would you lower it?
5. When would you pick `availableNow` over `processingTime` for a real pipeline, and what does that cost you?